# Model selection, classification & PCA: practice notebook

5 exercises, each with hidden tests. Data is loaded as **polars**; pandas only appears where a library insists on it.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("v1")` prints the reference solution.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, scipy, statsmodels, scikit-learn, matplotlib)
import importlib.util, subprocess, sys
for _p in ['polars']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The data every exercise uses. Read it if you like; don't change it.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

def rdata(name, package):
    """Load an R dataset (via statsmodels / Rdatasets) as a polars DataFrame, NaN → null."""
    df = pl.from_pandas(sm.datasets.get_rdataset(name, package, cache=True).data.reset_index(drop=True)).drop("rownames", strict=False)
    return df.with_columns(pl.col(pl.Float64).fill_nan(None))

auto = rdata("Auto", "ISLR")

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJ2MSI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLnBpcGVsaW5lIGltcG9ydCBtYWtlX3BpcGVsaW5lXG5mcm9tIHNrbGVhcm4ucHJlcHJvY2Vzc2luZyBpbXBvcnQgUG9seW5vbWlhbEZlYXR1cmVzXG5mcm9tIHNrbGVhcm4ubGluZWFyX21vZGVsIGltcG9ydCBMaW5lYXJSZWdyZXNzaW9uXG5mcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBLRm9sZCwgTGVhdmVPbmVPdXQsIGNyb3NzX3ZhbF9zY29yZVxuXG5kZWYgY3ZfZXJyb3JzKHNlZWQ9MCwga21heD0xMCk6XG4gICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpXG4gICAgeCA9IHJuZy51bmlmb3JtKDAsIDEsIDEwMClcbiAgICB5ID0gbnAuc2luKDkgKiB4KSArIHJuZy5ub3JtYWwoMCwgMC4yLCAxMDApXG4gICAgWCA9IHgucmVzaGFwZSgtMSwgMSlcbiAgICBvdXQgPSB7XCJrZm9sZFwiOiBbXSwgXCJsb29cIjogW119XG4gICAgZm9yIGsgaW4gcmFuZ2UoMSwga21heCArIDEpOlxuICAgICAgICBtb2RlbCA9IG1ha2VfcGlwZWxpbmUoUG9seW5vbWlhbEZlYXR1cmVzKGspLCBMaW5lYXJSZWdyZXNzaW9uKCkpXG4gICAgICAgIGZvciBuYW1lLCBjdiBpbiAoKFwia2ZvbGRcIiwgS0ZvbGQoNSwgc2h1ZmZsZT1UcnVlLCByYW5kb21fc3RhdGU9MCkpLCAoXCJsb29cIiwgTGVhdmVPbmVPdXQoKSkpOlxuICAgICAgICAgICAgb3V0W25hbWVdLmFwcGVuZCgtY3Jvc3NfdmFsX3Njb3JlKG1vZGVsLCBYLCB5LCBjdj1jdiwgc2NvcmluZz1cIm5lZ19tZWFuX3NxdWFyZWRfZXJyb3JcIikubWVhbigpKVxuICAgIHJldHVybiB7azogbnAuYXJyYXkodikgZm9yIGssIHYgaW4gb3V0Lml0ZW1zKCl9XG4iLCAidGVzdHMiOiAiXG5kZWYgY2xvc2UoYSwgYiwgcmVsPTFlLTYsIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGEgPSBmbG9hdChhKTsgYiA9IGZsb2F0KGIpXG4gICAgYXNzZXJ0IGFicyhhIC0gYikgPD0gcmVsICogbWF4KDEuMCwgYWJzKGIpKSwgZlwie3doYXR9ID0ge2E6LjZnfSwgZXhwZWN0ZWQge2I6LjZnfVwiXG5kZWYgYmV0d2VlbihhLCBsbywgaGksIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGFzc2VydCBsbyA8PSBmbG9hdChhKSA8PSBoaSwgZlwie3doYXR9ID0ge2Zsb2F0KGEpOi40Zn0sIGV4cGVjdGVkIGJldHdlZW4ge2xvfSBhbmQge2hpfVwiXG5kZWYgc2FtZV9rZXlzKGEsIGIsIHdoYXQ9XCJyZXN1bHRcIik6XG4gICAgYXNzZXJ0IGlzaW5zdGFuY2UoYSwgZGljdCksIGZcInt3aGF0fSBzaG91bGQgYmUgYSBkaWN0LCBnb3Qge3R5cGUoYSkuX19uYW1lX199XCJcbiAgICBhc3NlcnQgc2V0KGEpID09IHNldChiKSwgZlwie3doYXR9IGtleXMge3NvcnRlZChhKX0sIGV4cGVjdGVkIHtzb3J0ZWQoYil9XCJcblxuZGVmIF90KHNlZWQpOlxuICAgIGEgPSBVW1wiY3ZfZXJyb3JzXCJdKHNlZWQpOyBiID0gUltcImN2X2Vycm9yc1wiXShzZWVkKTsgc2FtZV9rZXlzKGEsIGIpXG4gICAgZm9yIGsgaW4gYjpcbiAgICAgICAgYXNzZXJ0IGxlbihhW2tdKSA9PSBsZW4oYltrXSksIGZcIntrfToge2xlbihhW2tdKX0gdmFsdWVzLCBleHBlY3RlZCB7bGVuKGJba10pfVwiXG4gICAgICAgIGZvciBpLCAoeCwgeSkgaW4gZW51bWVyYXRlKHppcChhW2tdLCBiW2tdKSk6IGNsb3NlKHgsIHksIDFlLTUsIGZcIntrfSBlcnJvciBmb3Igb3JkZXIge2kgKyAxfVwiKVxuVEVTVFMgPSBbKFwic2VlZCAwXCIsIGxhbWJkYTogX3QoMCkpLCAoXCJoaWRkZW46IHNlZWQgM1wiLCBsYW1iZGE6IF90KDMpKV1cbiIsICJsYW5nIjogInB5In0sICJ2MiI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLmxpbmVhcl9tb2RlbCBpbXBvcnQgTGFzc29cblxuU0lHTUEgPSBucC5mdWxsKCgxMCwgMTApLCAwLjIpICsgbnAuZXllKDEwKSAqIDEuOFxuQkVUQSA9IG5wLmFycmF5KFszLCAyLCAxLCAwLCAwLCAwLCAwLCAwLCAwLCAwLjBdKVxuXG5kZWYgbGFzc29fYmlhc192YXIoYWxwaGFzLCByZXBzPTIwMCwgc2VlZD0wKTpcbiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoc2VlZClcbiAgICBlc3QgPSBucC5lbXB0eSgocmVwcywgbGVuKGFscGhhcyksIDEwKSlcbiAgICBmb3IgciBpbiByYW5nZShyZXBzKTpcbiAgICAgICAgWCA9IHJuZy5tdWx0aXZhcmlhdGVfbm9ybWFsKG5wLnplcm9zKDEwKSwgU0lHTUEsIDEwMClcbiAgICAgICAgeSA9IFggQCBCRVRBICsgcm5nLm5vcm1hbChzaXplPTEwMClcbiAgICAgICAgZm9yIGksIGEgaW4gZW51bWVyYXRlKGFscGhhcyk6XG4gICAgICAgICAgICBlc3RbciwgaV0gPSBMYXNzbyhhbHBoYT1hKS5maXQoWCwgeSkuY29lZl9cbiAgICByZXR1cm4gZXN0Lm1lYW4oYXhpcz0wKSAtIEJFVEEsIGVzdC52YXIoYXhpcz0wKVxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuZGVmIHNhbWVfa2V5cyhhLCBiLCB3aGF0PVwicmVzdWx0XCIpOlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIGRpY3QpLCBmXCJ7d2hhdH0gc2hvdWxkIGJlIGEgZGljdCwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IHNldChhKSA9PSBzZXQoYiksIGZcInt3aGF0fSBrZXlzIHtzb3J0ZWQoYSl9LCBleHBlY3RlZCB7c29ydGVkKGIpfVwiXG5cbkFMID0gbnAuYXJyYXkoWzAuMDAxLCAwLjEsIDEuMF0pXG5kZWYgX3QoKTpcbiAgICBiLCB2ID0gKG5wLmFzYXJyYXkoeikgZm9yIHogaW4gVVtcImxhc3NvX2JpYXNfdmFyXCJdKEFMLCByZXBzPTE1MCwgc2VlZD0xKSlcbiAgICBhc3NlcnQgYi5zaGFwZSA9PSAoMywgMTApIGFuZCB2LnNoYXBlID09ICgzLCAxMCksIGZcInNoYXBlcyB7Yi5zaGFwZX0sIHt2LnNoYXBlfSwgZXhwZWN0ZWQgKDMsIDEwKSBlYWNoXCJcbiAgICBiZXR3ZWVuKGJbMCwgMF0sIC0wLjA2LCAwLjA2LCBcImJpYXMgb2YgXHUwM2IyMSBhdCBcdTAzYmIgPSAwLjAwMVwiKVxuICAgIGJldHdlZW4oYlsyLCAwXSwgLTEuNiwgLTAuNSwgXCJiaWFzIG9mIFx1MDNiMjEgYXQgXHUwM2JiID0gMVwiKVxuICAgIGFzc2VydCB2WzIsIDBdIDwgdlswLCAwXSwgXCJ2YXJpYW5jZSBvZiBcdTAzYjIxIHNob3VsZCBmYWxsIGFzIFx1MDNiYiBncm93c1wiXG4gICAgYmV0d2VlbihiWzIsIDVdLCAtMC4wNSwgMC4wNSwgXCJiaWFzIG9mIGEgemVybyBjb2VmZmljaWVudCBhdCBcdTAzYmIgPSAxXCIpXG5URVNUUyA9IFsoXCJiaWFzIGFuZCB2YXJpYW5jZSBiZWhhdmUgYXMgdGhlb3J5IHNheXNcIiwgX3QpXVxuIiwgImxhbmciOiAicHkifSwgInYzIjogeyJzb2x1dGlvbiI6ICJmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCB0cmFpbl90ZXN0X3NwbGl0XG5GRUFUVVJFUyA9IFtcImRpc3BsYWNlbWVudFwiLCBcImhvcnNlcG93ZXJcIiwgXCJ3ZWlnaHRcIiwgXCJhY2NlbGVyYXRpb25cIl1cblxuZGVmIGxvZ2l0X2F1dG8oZGYsIHNlZWQ9MCk6XG4gICAgWCA9IGRmLnNlbGVjdChGRUFUVVJFUykudG9fbnVtcHkoKS5hc3R5cGUoZmxvYXQpXG4gICAgeSA9IChkZltcIm1wZ1wiXSA+IGRmW1wibXBnXCJdLm1lZGlhbigpKS5jYXN0KHBsLkludDY0KS50b19udW1weSgpXG4gICAgWHRyLCBYdGUsIHl0ciwgeXRlID0gdHJhaW5fdGVzdF9zcGxpdChYLCB5LCB0ZXN0X3NpemU9MC4zLCByYW5kb21fc3RhdGU9c2VlZCwgc3RyYXRpZnk9eSlcbiAgICBtID0gc20uTG9naXQoeXRyLCBzbS5hZGRfY29uc3RhbnQoWHRyKSkuZml0KGRpc3A9MClcbiAgICBwcmVkID0gKG0ucHJlZGljdChzbS5hZGRfY29uc3RhbnQoWHRlKSkgPiAwLjUpLmFzdHlwZShpbnQpXG4gICAgZSA9IGZsb2F0KG5wLm1lYW4ocHJlZCAhPSB5dGUpKVxuICAgIGggPSAxLjk2ICogbnAuc3FydChlICogKDEgLSBlKSAvIGxlbih5dGUpKVxuICAgIHJldHVybiB7XCJlcnJvclwiOiBlLCBcImNpXCI6IChlIC0gaCwgZSArIGgpfVxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuZGVmIHNhbWVfa2V5cyhhLCBiLCB3aGF0PVwicmVzdWx0XCIpOlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIGRpY3QpLCBmXCJ7d2hhdH0gc2hvdWxkIGJlIGEgZGljdCwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IHNldChhKSA9PSBzZXQoYiksIGZcInt3aGF0fSBrZXlzIHtzb3J0ZWQoYSl9LCBleHBlY3RlZCB7c29ydGVkKGIpfVwiXG5cbmRlZiBfdChzZWVkKTpcbiAgICBhID0gVVtcImxvZ2l0X2F1dG9cIl0oYXV0bywgc2VlZCk7IGIgPSBSW1wibG9naXRfYXV0b1wiXShhdXRvLCBzZWVkKTsgc2FtZV9rZXlzKGEsIGIpXG4gICAgY2xvc2UoYVtcImVycm9yXCJdLCBiW1wiZXJyb3JcIl0sIDFlLTksIFwidGVzdCBlcnJvclwiKVxuICAgIGNsb3NlKGFbXCJjaVwiXVswXSwgYltcImNpXCJdWzBdLCAxZS02LCBcIkNJIGxvd1wiKTsgY2xvc2UoYVtcImNpXCJdWzFdLCBiW1wiY2lcIl1bMV0sIDFlLTYsIFwiQ0kgaGlnaFwiKVxuVEVTVFMgPSBbKFwic2VlZCAwXCIsIGxhbWJkYTogX3QoMCkpLCAoXCJoaWRkZW46IHNlZWQgNFwiLCBsYW1iZGE6IF90KDQpKV1cbiIsICJsYW5nIjogInB5In0sICJ2NCI6IHsic29sdXRpb24iOiAiZnJvbSBza2xlYXJuLm1vZGVsX3NlbGVjdGlvbiBpbXBvcnQgdHJhaW5fdGVzdF9zcGxpdFxuZnJvbSBza2xlYXJuLnBpcGVsaW5lIGltcG9ydCBtYWtlX3BpcGVsaW5lXG5mcm9tIHNrbGVhcm4ucHJlcHJvY2Vzc2luZyBpbXBvcnQgU3RhbmRhcmRTY2FsZXJcbmZyb20gc2tsZWFybi5saW5lYXJfbW9kZWwgaW1wb3J0IExvZ2lzdGljUmVncmVzc2lvblxuZnJvbSBza2xlYXJuLmRpc2NyaW1pbmFudF9hbmFseXNpcyBpbXBvcnQgTGluZWFyRGlzY3JpbWluYW50QW5hbHlzaXNcbmZyb20gc2tsZWFybi5lbnNlbWJsZSBpbXBvcnQgUmFuZG9tRm9yZXN0Q2xhc3NpZmllclxuZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IHJvY19hdWNfc2NvcmUsIFJvY0N1cnZlRGlzcGxheVxuRkVBVFVSRVMgPSBbXCJkaXNwbGFjZW1lbnRcIiwgXCJob3JzZXBvd2VyXCIsIFwid2VpZ2h0XCIsIFwiYWNjZWxlcmF0aW9uXCJdXG5cbmRlZiBjb21wYXJlX2NsYXNzaWZpZXJzKGRmLCBzZWVkPTApOlxuICAgIFggPSBkZi5zZWxlY3QoRkVBVFVSRVMpLnRvX251bXB5KCkuYXN0eXBlKGZsb2F0KVxuICAgIHkgPSAoZGZbXCJtcGdcIl0gPiBkZltcIm1wZ1wiXS5tZWRpYW4oKSkuY2FzdChwbC5JbnQ2NCkudG9fbnVtcHkoKVxuICAgIFh0ciwgWHRlLCB5dHIsIHl0ZSA9IHRyYWluX3Rlc3Rfc3BsaXQoWCwgeSwgdGVzdF9zaXplPTAuMywgcmFuZG9tX3N0YXRlPXNlZWQsIHN0cmF0aWZ5PXkpXG4gICAgbW9kZWxzID0ge1wibG9naXN0aWNcIjogbWFrZV9waXBlbGluZShTdGFuZGFyZFNjYWxlcigpLCBMb2dpc3RpY1JlZ3Jlc3Npb24oKSksXG4gICAgICAgICAgICAgIFwibGRhXCI6IExpbmVhckRpc2NyaW1pbmFudEFuYWx5c2lzKCksXG4gICAgICAgICAgICAgIFwiZm9yZXN0XCI6IFJhbmRvbUZvcmVzdENsYXNzaWZpZXIobl9lc3RpbWF0b3JzPTMwMCwgcmFuZG9tX3N0YXRlPTApfVxuICAgIG91dCA9IHt9XG4gICAgZm9yIG5hbWUsIG0gaW4gbW9kZWxzLml0ZW1zKCk6XG4gICAgICAgIG0uZml0KFh0ciwgeXRyKVxuICAgICAgICBvdXRbbmFtZV0gPSAoZmxvYXQobnAubWVhbihtLnByZWRpY3QoWHRlKSAhPSB5dGUpKSwgZmxvYXQocm9jX2F1Y19zY29yZSh5dGUsIG0ucHJlZGljdF9wcm9iYShYdGUpWzosIDFdKSkpXG4gICAgcmV0dXJuIG91dFxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuZGVmIHNhbWVfa2V5cyhhLCBiLCB3aGF0PVwicmVzdWx0XCIpOlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIGRpY3QpLCBmXCJ7d2hhdH0gc2hvdWxkIGJlIGEgZGljdCwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IHNldChhKSA9PSBzZXQoYiksIGZcInt3aGF0fSBrZXlzIHtzb3J0ZWQoYSl9LCBleHBlY3RlZCB7c29ydGVkKGIpfVwiXG5cbmRlZiBfdChzZWVkKTpcbiAgICBhID0gVVtcImNvbXBhcmVfY2xhc3NpZmllcnNcIl0oYXV0bywgc2VlZCk7IGIgPSBSW1wiY29tcGFyZV9jbGFzc2lmaWVyc1wiXShhdXRvLCBzZWVkKTsgc2FtZV9rZXlzKGEsIGIpXG4gICAgZm9yIGsgaW4gYjpcbiAgICAgICAgY2xvc2UoYVtrXVswXSwgYltrXVswXSwgMWUtOSwgayArIFwiIGVycm9yXCIpOyBjbG9zZShhW2tdWzFdLCBiW2tdWzFdLCAxZS02LCBrICsgXCIgQVVDXCIpXG5URVNUUyA9IFsoXCJzZWVkIDBcIiwgbGFtYmRhOiBfdCgwKSksIChcImhpZGRlbjogc2VlZCAyXCIsIGxhbWJkYTogX3QoMikpXVxuIiwgImxhbmciOiAicHkifSwgInY1IjogeyJzb2x1dGlvbiI6ICJmcm9tIHNrbGVhcm4ucHJlcHJvY2Vzc2luZyBpbXBvcnQgU3RhbmRhcmRTY2FsZXJcbmZyb20gc2tsZWFybi5kZWNvbXBvc2l0aW9uIGltcG9ydCBQQ0FcbkNPTFMgPSBbXCJtcGdcIiwgXCJjeWxpbmRlcnNcIiwgXCJkaXNwbGFjZW1lbnRcIiwgXCJob3JzZXBvd2VyXCIsIFwid2VpZ2h0XCIsIFwiYWNjZWxlcmF0aW9uXCIsIFwieWVhclwiXVxuXG5kZWYgYXV0b19wY2EoZGYpOlxuICAgIFogPSBTdGFuZGFyZFNjYWxlcigpLmZpdF90cmFuc2Zvcm0oZGYuc2VsZWN0KENPTFMpLnRvX251bXB5KCkuYXN0eXBlKGZsb2F0KSlcbiAgICBwID0gUENBKCkuZml0KFopXG4gICAgdiA9IHAuY29tcG9uZW50c19bMF1cbiAgICB2ID0gdiAqIG5wLnNpZ24odltucC5hcmdtYXgobnAuYWJzKHYpKV0pXG4gICAgcmV0dXJuIHtcInJhdGlvXCI6IHAuZXhwbGFpbmVkX3ZhcmlhbmNlX3JhdGlvXywgXCJwYzFcIjoge2M6IGZsb2F0KHgpIGZvciBjLCB4IGluIHppcChDT0xTLCB2KX19XG4iLCAidGVzdHMiOiAiXG5kZWYgY2xvc2UoYSwgYiwgcmVsPTFlLTYsIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGEgPSBmbG9hdChhKTsgYiA9IGZsb2F0KGIpXG4gICAgYXNzZXJ0IGFicyhhIC0gYikgPD0gcmVsICogbWF4KDEuMCwgYWJzKGIpKSwgZlwie3doYXR9ID0ge2E6LjZnfSwgZXhwZWN0ZWQge2I6LjZnfVwiXG5kZWYgYmV0d2VlbihhLCBsbywgaGksIHdoYXQ9XCJ2YWx1ZVwiKTpcbiAgICBhc3NlcnQgYSBpcyBub3QgTm9uZSwgXCJ5b3VyIGZ1bmN0aW9uIHJldHVybmVkIE5vbmUuIERpZCB5b3UgZm9yZ2V0IHRvIHJldHVybj9cIlxuICAgIGFzc2VydCBsbyA8PSBmbG9hdChhKSA8PSBoaSwgZlwie3doYXR9ID0ge2Zsb2F0KGEpOi40Zn0sIGV4cGVjdGVkIGJldHdlZW4ge2xvfSBhbmQge2hpfVwiXG5kZWYgc2FtZV9rZXlzKGEsIGIsIHdoYXQ9XCJyZXN1bHRcIik6XG4gICAgYXNzZXJ0IGlzaW5zdGFuY2UoYSwgZGljdCksIGZcInt3aGF0fSBzaG91bGQgYmUgYSBkaWN0LCBnb3Qge3R5cGUoYSkuX19uYW1lX199XCJcbiAgICBhc3NlcnQgc2V0KGEpID09IHNldChiKSwgZlwie3doYXR9IGtleXMge3NvcnRlZChhKX0sIGV4cGVjdGVkIHtzb3J0ZWQoYil9XCJcblxuZGVmIF90KGRmKTpcbiAgICBhID0gVVtcImF1dG9fcGNhXCJdKGRmKTsgYiA9IFJbXCJhdXRvX3BjYVwiXShkZik7IHNhbWVfa2V5cyhhLCBiKVxuICAgIGZvciBpLCAoeCwgeSkgaW4gZW51bWVyYXRlKHppcChhW1wicmF0aW9cIl0sIGJbXCJyYXRpb1wiXSkpOiBjbG9zZSh4LCB5LCAxZS02LCBmXCJyYXRpbyBvZiBQQ3tpICsgMX1cIilcbiAgICBzYW1lX2tleXMoYVtcInBjMVwiXSwgYltcInBjMVwiXSwgXCJwYzFcIilcbiAgICBmb3IgayBpbiBiW1wicGMxXCJdOiBjbG9zZShhW1wicGMxXCJdW2tdLCBiW1wicGMxXCJdW2tdLCAxZS02LCBcIlBDMSBsb2FkaW5nIG9mIFwiICsgaylcblRFU1RTID0gWyhcImF1dG9cIiwgbGFtYmRhOiBfdChhdXRvKSksIChcImhpZGRlbjogVVMgY2FycyBvbmx5XCIsIGxhbWJkYTogX3QoYXV0by5maWx0ZXIocGwuY29sKFwib3JpZ2luXCIpID09IDEpKSldXG4iLCAibGFuZyI6ICJweSJ9fQ==").decode())
_SETUP = _b64.b64decode("aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwb2xhcnMgYXMgcGwKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdAppbXBvcnQgc3RhdHNtb2RlbHMuYXBpIGFzIHNtCmZyb20gc2NpcHkgaW1wb3J0IHN0YXRzCgpkZWYgcmRhdGEobmFtZSwgcGFja2FnZSk6CiAgICAiIiJMb2FkIGFuIFIgZGF0YXNldCAodmlhIHN0YXRzbW9kZWxzIC8gUmRhdGFzZXRzKSBhcyBhIHBvbGFycyBEYXRhRnJhbWUsIE5hTiDihpIgbnVsbC4iIiIKICAgIGRmID0gcGwuZnJvbV9wYW5kYXMoc20uZGF0YXNldHMuZ2V0X3JkYXRhc2V0KG5hbWUsIHBhY2thZ2UsIGNhY2hlPVRydWUpLmRhdGEucmVzZXRfaW5kZXgoZHJvcD1UcnVlKSkuZHJvcCgicm93bmFtZXMiLCBzdHJpY3Q9RmFsc2UpCiAgICByZXR1cm4gZGYud2l0aF9jb2x1bW5zKHBsLmNvbChwbC5GbG9hdDY0KS5maWxsX25hbihOb25lKSkKCmF1dG8gPSByZGF0YSgiQXV0byIsICJJU0xSIik=").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. AoR 3.6 ex. 5: choose the polynomial order for sin(9x) by K-fold and leave-one-out CV
*Medium · id `v1`*

<p>Exercise 3.6.5. Write <code>cv_errors(seed=0, kmax=10)</code>: draw <code>x = rng.uniform(0, 1, 100)</code> then <code>y = np.sin(9 * x) + rng.normal(0, 0.2, 100)</code>. For each order k = 1..kmax fit <code>make_pipeline(PolynomialFeatures(k), LinearRegression())</code> and estimate the prediction error (MSE):</p><ul>
<li>5-fold: <code>KFold(5, shuffle=True, random_state=0)</code> (training size 4n/5, as in the book);</li>
<li>leave-one-out: <code>LeaveOneOut()</code>.</li></ul>
<p>Return a dict <code>{"kfold": array, "loo": array}</code>, each of length kmax. Plot both against k. Which order wins?</p>

**Theory to know cold**
<ul><li>Training error always falls with k; CV error is U-shaped. LOO is nearly unbiased but high-variance; K-fold trades a little bias for much less variance.</li><li><code>cross_val_score(..., scoring="neg_mean_squared_error")</code> returns negated MSEs.</li></ul>

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_score

def cv_errors(seed=0, kmax=10):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    e = cv_errors()
    plt.plot(range(1, 11), e["kfold"], "o-", label="5-fold"); plt.plot(range(1, 11), e["loo"], "s-", label="LOO")
    plt.yscale("log"); plt.xlabel("order k"); plt.legend(); plt.show()
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("v1")

---
## 2. AoR 4.7 ex. 6: lasso on correlated features: bias and variance of each βⱼ as λ changes
*Hard · id `v2`*

<p>Exercise 4.7.6. d = 10 features, X ~ N(0, Σ) with Σⱼⱼ = 2 and Σⱼₖ = 0.2; Y = 3X₁ + 2X₂ + X₃ + ε, ε ~ N(0, 1), n = 100.</p>
<p>Write <code>lasso_bias_var(alphas, reps=200, seed=0)</code>: repeat <code>reps</code> times (draw X with <code>rng.multivariate_normal</code>, then ε), fit <code>Lasso(alpha=a)</code> for every a, and return <code>(bias, var)</code>: two arrays of shape (len(alphas), 10) with the bias and variance of each β̂ⱼ.</p>
<p>Plot bias² and variance of β̂₁ against λ. Where is the trade-off best?</p>

**Theory to know cold**
<ul><li>The lasso shrinks every coefficient toward 0 by roughly λ (soft-thresholding when features are orthogonal), so large true βⱼ get bias ≈ −λ and small ones are set to exactly 0.</li><li>Variance falls as λ grows. Least squares is the λ = 0 end: unbiased, highest variance.</li></ul>

In [ ]:
from sklearn.linear_model import Lasso

SIGMA = np.full((10, 10), 0.2) + np.eye(10) * 1.8
BETA = np.array([3, 2, 1, 0, 0, 0, 0, 0, 0, 0.0])

def lasso_bias_var(alphas, reps=200, seed=0):
    rng = np.random.default_rng(seed)
    # your code here
    pass

# Sample run
try:
    al = np.array([0.001, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0])
    b, v = lasso_bias_var(al, reps=100)
    plt.plot(al, b[:, 0]**2, "o-", label="bias² β1"); plt.plot(al, v[:, 0], "s-", label="var β1")
    plt.xscale("log"); plt.legend(); plt.show()
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("v2")

---
## 3. AoR 5.6 ex. 2: Auto, logistic regression for above-median mpg, test error with a CI
*Medium · id `v3`*

<p>Exercise 5.6.2. Keep <code>mpg, displacement, horsepower, weight, acceleration</code>; set Y = 1 if mpg is above its median (strictly), else 0.</p>
<p>Write <code>logit_auto(df, seed=0)</code>: split with <code>train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)</code>, fit an unpenalised logistic regression with <code>sm.Logit</code> (add a constant) on the training set, classify the test set at 0.5, and return a dict <code>{"error": e, "ci": (lo, hi)}</code> with the normal-approximation 95% CI e ± 1.96 √(e(1−e)/n_test).</p>

**Theory to know cold**
<ul><li>The test error is a Binomial proportion, so its CI is the usual one for a proportion.</li><li>scikit-learn's <code>LogisticRegression</code> is ℓ2-penalised by default; statsmodels' <code>Logit</code> is plain maximum likelihood and gives SEs and p-values.</li></ul>

In [ ]:
from sklearn.model_selection import train_test_split
FEATURES = ["displacement", "horsepower", "weight", "acceleration"]

def logit_auto(df: pl.DataFrame, seed=0) -> dict:
    # your code here
    pass

# Sample run
try:
    print(logit_auto(auto))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("v3")

---
## 4. AoR 9.12 ex. 3 (b, c, e): Auto, logistic vs LDA vs random forest, ROC and AUC
*Medium · id `v4`*

<p>Exercise 9.12.3 with the same Y and features as the previous exercise and the same split (<code>test_size=0.3, random_state=seed, stratify=y</code>).</p>
<p>Write <code>compare_classifiers(df, seed=0)</code> fitting three scikit-learn models: <code>make_pipeline(StandardScaler(), LogisticRegression())</code>, <code>LinearDiscriminantAnalysis()</code>, <code>RandomForestClassifier(n_estimators=300, random_state=0)</code>. Return a dict <code>{"logistic": (error, auc), "lda": (...), "forest": (...)}</code> on the test set. Plot the three ROC curves with <code>RocCurveDisplay.from_estimator</code>.</p>

**Theory to know cold**
<ul><li>LDA assumes Gaussian classes with a shared covariance; when that holds it beats logistic regression with little data.</li><li>AUC measures ranking, error rate measures the 0.5-threshold decision; they can disagree.</li></ul>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, RocCurveDisplay
FEATURES = ["displacement", "horsepower", "weight", "acceleration"]

def compare_classifiers(df: pl.DataFrame, seed=0) -> dict:
    # your code here
    pass

# Sample run
try:
    print(compare_classifiers(auto))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("v4")

---
## 5. scikit-learn PCA on Auto's numeric columns: standardise, scree plot, read the loadings
*Medium · id `v5`*

<p>Write <code>auto_pca(df)</code> on the columns <code>mpg, cylinders, displacement, horsepower, weight, acceleration, year</code>: standardise with <code>StandardScaler</code>, fit <code>PCA()</code>, and return a dict with <code>"ratio"</code> (the explained-variance ratios, an array) and <code>"pc1"</code> (column → loading on the first component, with the sign flipped if needed so that the largest |loading| is positive).</p>
<p>Draw the scree plot. What does PC1 measure? Why standardise first?</p>

**Theory to know cold**
<ul><li>PCA on standardised data = eigendecomposition of the correlation matrix; ratios are eigenvalues / their sum.</li><li>Components are defined up to sign, so fix a convention before comparing.</li><li>Without scaling, weight (in pounds) would dominate PC1 just because its variance is largest.</li></ul>

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
COLS = ["mpg", "cylinders", "displacement", "horsepower", "weight", "acceleration", "year"]

def auto_pca(df: pl.DataFrame) -> dict:
    # your code here
    pass

# Sample run
try:
    r = auto_pca(auto)
    print(r["pc1"])
    plt.plot(range(1, 8), r["ratio"], "o-"); plt.xlabel("component"); plt.ylabel("explained variance ratio"); plt.show()
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("v5")